# GPU Parallelization of Casadi Functions

In [1]:
from cusadi.parallelization import parallelize_functions


# Setup parallelization parameters
BATCH_SIZE = 1024           # Number of function evaluations to parallelize
PRECISION = 'float'         # 'float' or 'double' (default: 'float')
FUNCTIONS = ['inertial_quantities']        # List of casadi function names to be parallelized (default: 'all')
DYNAMIC_BATCHING = True     # Enable dynamic batching (default: True)

# This function codegens and compiles JIT CUDA kernels for all specified Casadi functions.
# Regenerates and recompiles the functions even if no changes were made.
# Only needs to be called for modified or new functions (avoid needless recompilation).
parallelize_functions(fn_names=FUNCTIONS, batch_size=BATCH_SIZE,
                      precision=PRECISION, dynamic_batching=DYNAMIC_BATCHING)

Loaded CasADi function: inertial_quantities (8050 instructions)
Generating CUDA code for CasADi function:  inertial_quantities
     Dynamic batching:  True
     Number of instructions:  8050
     Number of inputs:  2
     Number of outputs:  4
     Number of work variables:  516
     Batch size:  1024
     Precision:  float
CUDA codegen complete for inertial_quantities.
Kernel written to /home/sehwan/Research/cusadi/cusadi/parallelization/codegen/inertial_quantities.cu
Pybind complete for inertial_quantities
Binding written to /home/sehwan/Research/cusadi/cusadi/parallelization/codegen/bindings.cpp

**********Compiling kernels for detected CUDA architecture 8.6**********
Loading JIT kernels from:  /home/sehwan/Research/cusadi/cusadi/parallelization/codegen
Build directory:  /home/sehwan/Research/cusadi/build


Detected CUDA files, patching ldflags
Emitting ninja build file /home/sehwan/Research/cusadi/build/build.ninja...
Building extension module cusadi_kernels...
Allowing ninja to set a default number of workers... (overridable by setting the environment variable MAX_JOBS=N)


[1/3] c++ -MMD -MF bindings.o.d -DTORCH_EXTENSION_NAME=cusadi_kernels -DTORCH_API_INCLUDE_EXTENSION_H -DPYBIND11_COMPILER_TYPE=\"_gcc\" -DPYBIND11_STDLIB=\"_libstdcpp\" -DPYBIND11_BUILD_ABI=\"_cxxabi1016\" -isystem /home/sehwan/miniconda3/envs/env_cusadi/lib/python3.11/site-packages/torch/include -isystem /home/sehwan/miniconda3/envs/env_cusadi/lib/python3.11/site-packages/torch/include/torch/csrc/api/include -isystem /usr/local/cuda/include -isystem /home/sehwan/miniconda3/envs/env_cusadi/include/python3.11 -D_GLIBCXX_USE_CXX11_ABI=1 -fPIC -std=c++17 -O3 -march=native -c /home/sehwan/Research/cusadi/cusadi/parallelization/codegen/bindings.cpp -o bindings.o 
[2/3] /usr/local/cuda/bin/nvcc --generate-dependencies-with-compile --dependency-output inertial_quantities.cuda.o.d -DTORCH_EXTENSION_NAME=cusadi_kernels -DTORCH_API_INCLUDE_EXTENSION_H -DPYBIND11_COMPILER_TYPE=\"_gcc\" -DPYBIND11_STDLIB=\"_libstdcpp\" -DPYBIND11_BUILD_ABI=\"_cxxabi1016\" -isystem /home/sehwan/miniconda3/envs/env_

Loading extension module cusadi_kernels...


In [3]:
import os
import casadi as ca
from cusadi import FUNCTION_DIR
from cusadi.parallelization import CusadiFunction

# Builds CUDA kernels with bindings and loads JIT libraries for use with CusadiFunction.
test_casadi_fn = ca.Function.load(os.path.join(FUNCTION_DIR, 'inertial_quantities.casadi'))
test_cusadi_fn = CusadiFunction(test_casadi_fn, batch_size=10000)
test_cusadi_fn.test(10000)

Loaded CasADi function inertial_quantities with 8050 instructions.
Loaded library:  <built-in method inertial_quantities of PyCapsule object at 0x7ce5b13e9980>
Checking input dimensions...
    Input tensor sizes:  [torch.Size([10000, 24]), torch.Size([10000, 24])]
    Output tensor sizes:  [torch.Size([10000, 3]), torch.Size([10000, 36]), torch.Size([10000, 6]), torch.Size([10000, 144])]
    Work tensor size:  torch.Size([516, 10000])
Time taken for 10000 environments (GPU): 0.008635659 seconds.
Time taken for 10000 environments (serial, CPU): 2.93740493 seconds.
Average error for each environment:
    Output 0: Average error norm/env for 10000 envs.: 1.0558850517557028e-08
    Output 1: Average error norm/env for 10000 envs.: 2.1330998282609134e-07
    Output 2: Average error norm/env for 10000 envs.: 1.663327134457138e-06
    Output 3: Average error norm/env for 10000 envs.: 1.1479158799982071e-07
